## Cell 1 — Imports & config  (unchanged)

In [ ]:
import requests, urllib3
import pytz

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

file_date  = "2026-06-02"
url        = "<YOUR_ELASTICSEARCH_URL>/_search"
user       = "<USERNAME>"
psd        = "<PASSWORD>"
headers    = {"Content-Type": "application/json"}
all_scopes = ['SWCC', 'TDCC', 'PBCC', 'PTCC']

kolkata_tz      = pytz.timezone("Asia/Kolkata")
INTERVAL_MIN    = 10
total_intervals = (24 * 60) // INTERVAL_MIN   # 144

## Cell 2 — Fetch loop  (builds `all_hits` from the Elasticsearch source)

Queries the source in `INTERVAL_MIN`-minute slices across the whole day and
accumulates every hit into `all_hits`, which the windowed loader below consumes.


In [ ]:
import json, time
from datetime import datetime, timedelta

# Anchor the day in Kolkata time so window boundaries line up with @timestamp
day_start = kolkata_tz.localize(datetime.strptime(file_date, "%Y-%m-%d"))

# One requests session for the whole day (basic auth + headers reused)
session         = requests.Session()
session.auth    = (user, psd)
session.headers.update(headers)
session.verify  = False   # matches verify=False in the source notebook

all_hits = []

# Walk the day in INTERVAL_MIN-minute slices. Each slice stays under the
# 10k result-window limit, so we never need scroll/search_after.
for i in range(total_intervals):
    win_start = day_start + timedelta(minutes=i * INTERVAL_MIN)
    win_end   = win_start + timedelta(minutes=INTERVAL_MIN)

    query = {
        "size": 10000,
        "_source": False,
        "fields": ["*"],
        "query": {
            "bool": {
                "filter": [
                    {"terms": {"scope.keyword": all_scopes}},
                    {"range": {"@timestamp": {
                        "gte": win_start.isoformat(),
                        "lt":  win_end.isoformat()
                    }}}
                ]
            }
        }
    }

    resp = session.post(url, data=json.dumps(query), timeout=60)
    resp.raise_for_status()
    hits = resp.json().get("hits", {}).get("hits", [])
    all_hits.extend(hits)

    print(f"Interval {i + 1:>3}/{total_intervals}  "
          f"[{win_start:%H:%M}\u2013{win_end:%H:%M}]:  "
          f"{len(hits):>6,} hits   (cumulative {len(all_hits):,})")

print(f"\nTotal hits fetched from source: {len(all_hits):,}")


In [ ]:
import json
from datetime import datetime
from pyspark.sql import functions as F

WINDOW_MIN = 30  # minutes per window

# Group all_hits by 30-min window using @timestamp
windows = {}
for record in all_hits:
    fields  = record.get("fields", {})
    ts_raw  = fields.get("@timestamp", [None])
    ts_str  = ts_raw[0] if isinstance(ts_raw, list) else ts_raw
    if ts_str:
        dt      = datetime.fromisoformat(ts_str[:16])
        win_key = (dt.hour * 60 + dt.minute) // WINDOW_MIN   # 0–47
    else:
        win_key = -1
    windows.setdefault(win_key, []).append(record)

total_windows = 24 * 60 // WINDOW_MIN   # 48

# Build one DF per window, union them — driver serialises ~19k records at a time (not 9 lakh).
# Union is lazy in Spark, so the final df is processed in chunks — no OOM.
# df is available after the loop so display(df) works normally.
df = None

for win_key in sorted(windows.keys()):
    records   = windows[win_key]
    start_min = win_key * WINDOW_MIN
    h, m      = divmod(start_min, 60)
    print(f"Window {win_key + 1}/{total_windows}  [{h:02d}:{m:02d} – {h:02d}:{m + WINDOW_MIN - 1:02d}]:  {len(records):,} records")

    flattened_records = []
    for record in records:
        flat = {}
        for key, value in record["fields"].items():
            flat[key] = value[0] if isinstance(value, list) and len(value) > 0 else value
        flattened_records.append(flat)

    window_df = spark.createDataFrame(flattened_records)
    df = window_df if df is None else df.union(window_df)

    del window_df, flattened_records, records
    windows[win_key] = None

print(f"total columns: {len(df.columns)}")

In [ ]:
# ── See the data from the source ─────────────────────────────────────────────
# `df` is built by the windowed loader above (union of all 30-min windows).
print(f"rows: {df.count():,}   columns: {len(df.columns)}")
display(df)          # rich, scrollable Databricks grid
# df.show(20, truncate=False)   # plain-text fallback outside Databricks


## Cell 3 — DataFrame creation  (**CHANGED** — 30-minute windows)

**Before:** `spark.createDataFrame(all_flattened)` — loads all ~9 lakh records at once → driver OOM  
**After:** groups `all_hits` by 30-min window using `@timestamp`, creates one small DataFrame per window, then frees memory

In [ ]:
import json
from datetime import datetime
from pyspark.sql import functions as F

WINDOW_MIN = 30  # size of each window in minutes

# ── Group all_hits by 30-min window using @timestamp ─────────────────────────
windows = {}
for record in all_hits:
    fields  = record.get("fields", {})
    ts_raw  = fields.get("@timestamp", [None])
    ts_str  = ts_raw[0] if isinstance(ts_raw, list) else ts_raw
    if ts_str:
        dt      = datetime.fromisoformat(ts_str[:16])          # "2026-06-02T00:09"
        win_key = (dt.hour * 60 + dt.minute) // WINDOW_MIN     # 0 – 47
    else:
        win_key = -1
    windows.setdefault(win_key, []).append(record)

total_windows = 24 * 60 // WINDOW_MIN   # 48

# ── Process one 30-min window at a time ──────────────────────────────────────
for win_key in sorted(windows.keys()):
    records   = windows[win_key]
    start_min = win_key * WINDOW_MIN
    h, m      = divmod(start_min, 60)
    print(f"\nWindow {win_key + 1}/{total_windows}  [{h:02d}:{m:02d} – {h:02d}:{m + WINDOW_MIN - 1:02d}]:  {len(records):,} records")

    # flatten fields (same logic as before)
    flattened_records = []
    for record in records:
        flat = {}
        for key, value in record["fields"].items():
            flat[key] = value[0] if isinstance(value, list) and len(value) > 0 else value
        flattened_records.append(flat)

    df = spark.createDataFrame(flattened_records)
    print(f"  total columns: {len(df.columns)}")

    # ── add your transformations / writes here ────────────────────────────────
    # e.g. df.write.mode("append").parquet(f"/mnt/output/{file_date}/window_{win_key+1:02d}")
    # ─────────────────────────────────────────────────────────────────────────

    # free this window's memory before moving to the next
    del df, flattened_records, records
    windows[win_key] = None